# Notebook 03 — Probability Fundamentals

**Module:** 03 — Statistics and Probability  
**Notebook:** 03 of 18  
**Prerequisites:** NB02  
**Time estimate:** 90 minutes

---
## Step 1 — Motivation

Probability is the foundation of inferential statistics. A p-value is a probability.
A confidence interval is defined using probability. Bayesian inference (NB13) requires
Bayes' theorem directly. Understanding conditional probability — $P(A|B)$ — is
essential for interpreting diagnostic tests, genome-wide association studies,
and any result where the base rate matters.

---
## Step 2 — Intuition

Probability quantifies uncertainty about events. $P(A) = 0.3$ means: if you ran
the experiment many times under identical conditions, event $A$ would occur about
30% of the time.

Conditional probability $P(A|B)$ asks: given that $B$ already happened, how likely
is $A$? This is almost always different from the unconditional $P(A)$.

Bayes' theorem is the rule for updating beliefs: start with a prior, observe data,
arrive at a posterior. The math is one line; the implications are profound.

---
## Step 3 — Biological Background

**Diagnostic test example:**

A genetic test for disease $D$ has:
- **Sensitivity** = $P(\text{test}+|D) = 0.95$ (true positive rate)
- **Specificity** = $P(\text{test}-|\bar{D}) = 0.99$ (true negative rate)
- **Prevalence** = $P(D) = 0.001$ (disease is rare)

What is $P(D|\text{test}+)$ — the probability you actually have the disease given a
positive test? Bayes' theorem gives the answer, and it surprises most people.

**Sequence motif probability:**
The probability of observing a specific 6-mer in a random DNA sequence is $(1/4)^6$.
Independence: each base is assumed i.i.d. Uniform(ACGT).

**Hardy-Weinberg equilibrium (Module 05):**
If alleles are independent, genotype frequencies = product of allele frequencies.
This is a direct application of the probability independence rule.

---
## Step 4 — Mathematical Explanation

**Probability axioms (Kolmogorov):**
1. $P(A) \geq 0$ for all events $A$
2. $P(\Omega) = 1$ (the sample space has probability 1)
3. If $A \cap B = \emptyset$: $P(A \cup B) = P(A) + P(B)$ (additivity)

**Derived rules:**
$$P(A^c) = 1 - P(A)$$
$$P(A \cup B) = P(A) + P(B) - P(A \cap B)$$

**Conditional probability:**
$$P(A|B) = \frac{P(A \cap B)}{P(B)}, \quad P(B) > 0$$

**Independence:** $A \perp B \iff P(A \cap B) = P(A) \cdot P(B)$

**Total probability:**
$$P(B) = P(B|A)P(A) + P(B|A^c)P(A^c)$$

**Bayes' theorem:**
$$P(A|B) = \frac{P(B|A) \cdot P(A)}{P(B)}$$

For the diagnostic test:
$$P(D|+) = \frac{P(+|D) \cdot P(D)}{P(+|D)P(D) + P(+|\bar{D})P(\bar{D})}$$

---
## Step 5 — Computational Explanation

Probability calculations in Python:
- **By formula:** implement directly from the mathematical definition
- **By simulation:** run the experiment many times with `np.random.default_rng`, count outcomes

Simulation is the sanity-check for formula results — if they don't agree, the formula
has an error. For Bayes' theorem problems, simulation via frequency tables is often
more intuitive than algebra.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Cell 6.1 — Bayes' theorem: diagnostic test
def bayes_diagnostic(sensitivity: float, specificity: float, prevalence: float) -> dict:
    """
    Compute positive and negative predictive values via Bayes' theorem.

    Parameters
    ----------
    sensitivity : P(test+ | disease)  — true positive rate
    specificity : P(test- | no disease) — true negative rate
    prevalence  : P(disease)

    Returns
    -------
    dict with PPV, NPV, and intermediate values
    """
    p_disease = prevalence
    p_no_disease = 1 - prevalence
    p_pos_given_disease = sensitivity
    p_pos_given_no_disease = 1 - specificity  # false positive rate

    # P(test+) by total probability
    p_pos = p_pos_given_disease * p_disease + p_pos_given_no_disease * p_no_disease

    # PPV = P(disease | test+) by Bayes' theorem
    ppv = (p_pos_given_disease * p_disease) / p_pos

    # NPV = P(no disease | test-)
    p_neg = 1 - p_pos
    p_neg_given_no_disease = specificity
    npv = (p_neg_given_no_disease * p_no_disease) / p_neg

    return dict(PPV=ppv, NPV=npv, P_test_positive=p_pos)

# Disease scenario
result = bayes_diagnostic(sensitivity=0.95, specificity=0.99, prevalence=0.001)
print("Diagnostic test: sensitivity=95%, specificity=99%, prevalence=0.1%")
print(f"  P(test positive) = {result['P_test_positive']:.4f}")
print(f"  PPV = P(disease | test+) = {result['PPV']:.4f}  ← most people are surprised by this")
print(f"  NPV = P(no disease | test-) = {result['NPV']:.6f}")
print(f"\nInterpretation: even with a 95% sensitive test, a positive result")
print(f"only means ~{result['PPV']*100:.1f}% chance of disease when disease is rare.")

In [ ]:
# Cell 6.2 — Verify with simulation (frequency approach)
rng = np.random.default_rng(42)
N = 1_000_000

disease = rng.random(N) < 0.001  # 0.1% prevalence
test_pos = np.where(
    disease,
    rng.random(N) < 0.95,   # sensitivity = 0.95
    rng.random(N) < 0.01,   # false positive rate = 1 - specificity
)

# P(disease | test+)
ppv_sim = disease[test_pos].mean()
print(f"Simulation PPV (N={N:,}): {ppv_sim:.4f}")
print(f"Formula PPV:              {result['PPV']:.4f}")
print(f"Agreement: {'YES' if abs(ppv_sim - result['PPV']) < 0.005 else 'NO'}")

In [ ]:
# Cell 6.3 — Independence: random DNA motif probability
def prob_motif_by_chance(motif_length: int, genome_length: int,
                          p_each_base: float = 0.25) -> float:
    """
    Expected number of times a specific k-mer appears by chance in a genome,
    assuming i.i.d. uniform base composition.
    """
    p_motif = p_each_base ** motif_length
    n_positions = genome_length - motif_length + 1
    return p_motif * n_positions

k_values = [4, 6, 8, 10]
L_genome = 3_000_000  # E. coli ~3 Mb
print(f"Expected k-mer occurrences in E. coli (~{L_genome/1e6:.0f} Mb, uniform sequence):")
print(f"{'k':<6} {'P(motif)':<16} {'Expected count':<18}")
print("-" * 42)
for k in k_values:
    p = 0.25**k
    exp = prob_motif_by_chance(k, L_genome)
    print(f"  {k}     {p:.2e}         {exp:.1f}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — PPV vs. prevalence: the base rate effect
prevalences = np.linspace(0.0001, 0.20, 200)
ppvs_95_99 = [bayes_diagnostic(0.95, 0.99, p)['PPV'] for p in prevalences]
ppvs_95_95 = [bayes_diagnostic(0.95, 0.95, p)['PPV'] for p in prevalences]
ppvs_99_99 = [bayes_diagnostic(0.99, 0.99, p)['PPV'] for p in prevalences]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(prevalences * 100, ppvs_95_99, lw=2, color="steelblue", label="Sens=95%, Spec=99%")
ax.plot(prevalences * 100, ppvs_95_95, lw=2, color="orange", label="Sens=95%, Spec=95%")
ax.plot(prevalences * 100, ppvs_99_99, lw=2, color="tomato", label="Sens=99%, Spec=99%")
ax.axvline(0.1, color="gray", ls="--", lw=1, label="Prevalence=0.1%")
ax.set_xlabel("Disease prevalence (%)"); ax.set_ylabel("PPV = P(disease | test+)")
ax.set_title("Bayes' theorem: PPV depends critically on prevalence")
ax.legend(fontsize=8)

# Panel 2: Venn-like frequency diagram for the diagnostic test
ax = axes[1]
N_pop = 100_000
n_disease = int(N_pop * 0.001)
n_no_disease = N_pop - n_disease
tp = int(n_disease * 0.95)
fn = n_disease - tp
fp = int(n_no_disease * 0.01)
tn = n_no_disease - fp

table_data = [[tp, fn, n_disease],
              [fp, tn, n_no_disease],
              [tp+fp, fn+tn, N_pop]]

rows = ["Disease", "No disease", "Total"]
cols = ["Test+", "Test-", "Total"]
ax.axis("off")
tbl = ax.table(cellText=[[f"{v:,}" for v in row] for row in table_data],
               rowLabels=rows, colLabels=cols,
               cellLoc="center", loc="center")
tbl.scale(1.2, 2)
ax.set_title(f"Confusion matrix (N={N_pop:,}, prev=0.1%, sens=95%, spec=99%)\n"
             f"PPV = {tp}/{tp+fp} = {tp/(tp+fp):.3f}")

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. A GWAS study identifies a SNP with $P(\text{test}+|\text{risk allele}) = 0.60$ and
   $P(\text{test}+|\text{no risk allele}) = 0.40$. If the risk allele frequency is 0.15,
   what is $P(\text{risk allele}|\text{test}+)$? Apply Bayes' theorem.
2. Two restriction sites are independent on a chromosome with uniform base composition.
   EcoRI cuts at GAATTC (6-mer). BamHI cuts at GGATCC (6-mer). What is the probability
   that both sites occur within 100 bp of each other in a random 10 kb region?
3. Implement `bayes_diagnostic()` from scratch without looking at Cell 6.1. Verify it
   gives the same result for sensitivity=0.9, specificity=0.95, prevalence=0.05.
4. The simulation in Cell 6.2 gives a slightly different PPV each run (if the seed
   changes). Compute the 95% confidence interval on the simulated PPV using the
   standard error of a proportion: $\text{SE} = \sqrt{p(1-p)/n}$.

---
## Quiz — Active Recall

1. Write Bayes' theorem from memory.
2. What is the formula for conditional probability $P(A|B)$?
3. Why is the PPV of a test low when disease prevalence is low, even if the test is
   accurate? Explain without equations.
4. What does independence of events mean mathematically?
5. Write the law of total probability for two events $A$ and $A^c$.

---
## Reflection

**Date completed:** ____________________

> *[Could you derive the diagnostic test PPV on a whiteboard? What is the base rate fallacy?]*

---
*Next: `04_distributions_normal_and_clt.ipynb`*